# Phase 2c — Accuracy Confirmation
**FSQA** | fine-tune pipeline

> **Bring your own data.** No data ships with this repo. This notebook reads `./local-data/phase1_fsqa_eval/<run id>/questions.jsonl` (JSON Lines, one question per line, as Phase 1 writes it); `./local-data/phase2_splits/split_<Phase 1 run id>.json` (the A/B split Phase 2b writes); and `./local-data/phase2b_screen/<2b run id>/manifest.json` (the Phase 2b manifest, for the question ids 2b used). With the run ids set to `None` it uses the last run folder in name order. Run Phases 1 and 2b first, or supply your own files in those formats. Each question record needs `id`, `question`, `reference_answer`, `domain`, `difficulty` and `seed_section`, and `citation` is read if present; the split file needs `set_a` and `set_b` (lists of question ids), and the manifest needs `sampled_question_ids`.

Narrows the 2b shortlist to a teacher for Phase 4. Two things change from 2b:

**1. The judge reads the actual CFR section, not a summary of it.**
2b graded against a 120-word reference answer, which is itself a paraphrase.
Here the full regulation text goes into every judge prompt. The judge checks
the answer against the law, and the reference becomes a second opinion rather
than the ground truth.

**2. Scoring is weighted toward accuracy.**
The winner generates 15,000 training pairs. A pair that is thin is
recoverable; a pair that is wrong gets baked into the student permanently.
accuracy is 0-5 (widened from 0-4 for finer separation), weighted 70%.
citation is 30%. completeness is recorded but does not score. contradiction
is a reported rate, not a modifier.

**Questions:** set A minus everything 2b used. Set B stays sealed for the final evaluation.

**Checkpointing:** every eval answer and every grading is appended to the
volume as it completes. A crash costs only the in-flight calls; re-running
this notebook resumes where it stopped.

In [ ]:
import concurrent.futures
import json
import os
import random
import re
import statistics
import threading
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import requests

print("imports ok", flush=True)

## Config

In [ ]:
# --- Shortlist from 2b ----------------------------------------------------
# An example shortlist. Replace it with the configs you picked from your own 2b ranking
# (tied with the leader on score, cheapest first), and keep PRICE in step with it.
CONFIGS = [
    {"config_id": "gpt-5.6-sol|xhigh",  "model": "openai/gpt-5.6-sol", "effort": "xhigh",  "mode": None},
    {"config_id": "gpt-5.6-sol|high",   "model": "openai/gpt-5.6-sol", "effort": "high",   "mode": None},
    {"config_id": "gpt-5.5|xhigh",      "model": "openai/gpt-5.5",     "effort": "xhigh",  "mode": None},
    {"config_id": "gpt-5.5|high",       "model": "openai/gpt-5.5",     "effort": "high",   "mode": None},
    {"config_id": "gpt-5.5|medium",     "model": "openai/gpt-5.5",     "effort": "medium", "mode": None},
]

JUDGES = {
    "gemini": {"slug": "google/gemini-3.1-pro-preview",
               "provider": {"order": ["google-vertex", "google-ai-studio"], "allow_fallbacks": True}},
    "kimi":   {"slug": "moonshotai/kimi-k3",
               "provider": {"order": ["digitalocean"], "allow_fallbacks": False}},
}
USE_JUDGES = ["gemini", "kimi"]

EVAL_PROVIDER = {"order": ["openai"], "allow_fallbacks": False}

# OpenRouter list prices, USD per million tokens (input, output). Each figure is the highest price listed across
# the model's OpenRouter endpoints, tiers and context bands (an upper bound). Source for each model:
# https://openrouter.ai/api/v1/models/<slug>/endpoints, retrieved 2026-11-05.
# A model with no confirmed price must NOT be given a guessed one: leave it out and the run refuses to start.
PRICE = {
    "openai/gpt-5.5":     (25.00, 112.50),
    "openai/gpt-5.6-sol": (8.80, 33.00),
    "google/gemini-3.1-pro-preview": (7.20, 32.40),
    "moonshotai/kimi-k3": (4.50, 22.50),
}

# --- Scoring weights ------------------------------------------------------
W_ACCURACY = 0.70
W_CITATION = 0.30
ACC_MAX, CITE_MAX = 5, 3

# --- Paths ----------------------------------------------------------------
VOLUME_MOUNT = "./local-data"
QUESTIONS_SUBDIR = "phase1_fsqa_eval"
SPLIT_DIR = "phase2_splits"
SCREEN_SUBDIR = "phase2b_screen"
OUTPUT_SUBDIR = "phase2c_confirm"

SCREEN_RUN_ID = None        # None = most recent 2b run
QUESTION_RUN_ID = None      # None = most recent Phase 1 run

# --- eCFR (for full-text judging) ----------------------------------------
ECFR_BASE = "https://www.ecfr.gov/api/versioner/v1/full"
ECFR_DATE = "2026-08-01"
ECFR_HEADERS = {"User-Agent": "labs-benchmark/1.0 (research; you@example.com)"}
CFR_TEXT_CAP = 15000        # chars of section text passed to the judge

# --- Execution ------------------------------------------------------------
EVAL_WORKERS = 12
JUDGE_WORKERS = 24
REQUEST_TIMEOUT = 300
MAX_RETRIES = 3
MAX_TOKENS = 32000
SEED = 20260825

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY missing. Attach 'your-modal-secret'.")
if not os.path.isdir(VOLUME_MOUNT):
    raise RuntimeError(f"{VOLUME_MOUNT} not mounted.")

HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
           "X-OpenRouter-Title": "labs-benchmark"}
random.seed(SEED)

# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's listed price in
# USD per million tokens (input, output). Entries the repo does not state are left for the operator to fill.
from datetime import datetime, timezone
import math
import os

# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_positive_float(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


# Default 25.00. Override with the environment variable RUN_BUDGET_USD (a finite number above 0, else the run refuses to start).
RUN_BUDGET_USD = _env_positive_float("RUN_BUDGET_USD", 25.00)

# Spend ledger and runner lock root, shared by every notebook: SPEND_LEDGER_DIR, default <repo root>/spend_ledger.
# On Modal there is no git checkout, so set SPEND_LEDGER_DIR to a path on the volume. Run ledgers live in
# <root>/<RUN_ID>/spend_ledger.jsonl and the single runner lock is <root>/runner.lock.
# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


SPEND_ROOT = _ledger_root()
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
missing_prices = sorted(set([c["model"] for c in CONFIGS] + [JUDGES[j]["slug"] for j in USE_JUDGES]) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"No confirmed price for model(s) {missing_prices}. Nothing was sent. Look up the listed "
                       "OpenRouter price (USD per million tokens, input and output) and add it to PRICE in the config "
                       "cell with its source URL and date, then rerun.")
for _m, _p in PRICE.items():
    if not (len(_p) == 2 and all(isinstance(x, (int, float)) and math.isfinite(x) and x > 0 for x in _p)):
        raise RuntimeError(f"PRICE entry for {_m} is not a pair of finite positive numbers. Nothing was sent.")
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
RUN_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_DIR = os.path.join(SPEND_ROOT, RUN_ID)
os.makedirs(LEDGER_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(LEDGER_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. An exclusive lock file at SPEND_ROOT/runner.lock: a second runner
# refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died without exiting)
# is never cleared automatically: confirm no runner is alive, then delete SPEND_ROOT/runner.lock by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(SPEND_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be using this run, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


print(f"Configs : {len(CONFIGS)}")
for c in CONFIGS:
    print(f"  {c['config_id']}")
print(f"Judges  : {USE_JUDGES}  (grading against full CFR text)")
print(f"Workers : eval {EVAL_WORKERS}, judge {JUDGE_WORKERS}")
print(f"Run cap : ${RUN_BUDGET_USD:.2f}   ledger: {LEDGER_PATH}")
print(f"Weights : accuracy {W_ACCURACY:.0%} / citation {W_CITATION:.0%}")

## Questions — set A minus everything 2b used

In [ ]:
def list_runs(subdir, required_file):
    """Directories that hold a completed run. Skips _checkpoint and any
    partial directory missing its output file."""
    base = os.path.join(VOLUME_MOUNT, subdir)
    return sorted(
        d for d in os.listdir(base)
        if not d.startswith("_")
        and os.path.isfile(os.path.join(base, d, required_file))
    )


runs = list_runs(QUESTIONS_SUBDIR, "questions.jsonl")
rid = QUESTION_RUN_ID or runs[-1]
ALL_Q = [json.loads(l) for l in open(
    os.path.join(VOLUME_MOUNT, QUESTIONS_SUBDIR, rid, "questions.jsonl"))]

sp = json.load(open(os.path.join(VOLUME_MOUNT, SPLIT_DIR, f"split_{rid}.json")))
A_IDS, B_IDS = set(sp["set_a"]), set(sp["set_b"])

screen_runs = list_runs(SCREEN_SUBDIR, "manifest.json")
srid = SCREEN_RUN_ID or screen_runs[-1]
screen_manifest = json.load(open(
    os.path.join(VOLUME_MOUNT, SCREEN_SUBDIR, srid, "manifest.json")))
USED_IN_2B = set(screen_manifest["sampled_question_ids"])

EVAL_QUESTIONS = [q for q in ALL_Q if q["id"] in A_IDS and q["id"] not in USED_IN_2B]

print(f"Phase 1 run : {rid}  ({len(ALL_Q)} questions)")
print(f"2b run      : {srid}  (used {len(USED_IN_2B)})")
print(f"Set A       : {len(A_IDS)}")
print(f"Set B       : {len(B_IDS)}  <- sealed, not loaded")
print(f"\nAvailable for 2c: {len(EVAL_QUESTIONS)}")
print(f"  by domain     : {dict(Counter(q['domain'] for q in EVAL_QUESTIONS))}")
print(f"  by difficulty : {dict(Counter(q['difficulty'] for q in EVAL_QUESTIONS))}")
print(f"  sections      : {len(set(q['seed_section'] for q in EVAL_QUESTIONS))}")

assert not any(q["id"] in B_IDS for q in EVAL_QUESTIONS), "Holdout leak: set B id in 2c"
assert not any(q["id"] in USED_IN_2B for q in EVAL_QUESTIONS), "Reused a 2b question"
print("\nIntegrity: no set B ids, no 2b reuse.")

## Fetch CFR section text

Phase 1 saved headings and char counts but not the text itself, so it gets
pulled fresh. Free, and it is what makes this a stricter accuracy test.

In [ ]:
def fetch_cfr_section(section, retries=4):
    part = section.split(".")[0]
    url = f"{ECFR_BASE}/{ECFR_DATE}/title-21.xml"
    for attempt in range(retries):
        try:
            r = requests.get(url, params={"part": part, "section": section},
                             timeout=60, headers=ECFR_HEADERS)
            if r.status_code == 200:
                r.encoding = "utf-8"
                txt = re.sub(r"<[^>]+>", " ", r.text)
                return re.sub(r"\s+", " ", txt).strip()
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** (attempt + 2))
                continue
            return None
        except Exception:
            time.sleep(2 ** attempt)
    return None


NEEDED = sorted({q["seed_section"] for q in EVAL_QUESTIONS})
print(f"Fetching {len(NEEDED)} sections...", flush=True)
CFR_TEXT = {}
for i, sec in enumerate(NEEDED, 1):
    print(f"  [{i}/{len(NEEDED)}] {sec}", flush=True)
    t = fetch_cfr_section(sec)
    if t:
        CFR_TEXT[sec] = t
    time.sleep(0.3)

missing = [s for s in NEEDED if s not in CFR_TEXT]
print(f"\nRetrieved {len(CFR_TEXT)}/{len(NEEDED)}")
if missing:
    print(f"MISSING: {missing}")
    EVAL_QUESTIONS = [q for q in EVAL_QUESTIONS if q["seed_section"] in CFR_TEXT]
    print(f"Questions after dropping unfetchable sections: {len(EVAL_QUESTIONS)}")
print(f"Total CFR text: {sum(len(v) for v in CFR_TEXT.values()):,} chars")
print(f"Sections over the {CFR_TEXT_CAP:,}-char cap (will truncate): "
      f"{[s for s,v in CFR_TEXT.items() if len(v) > CFR_TEXT_CAP]}")

## Checkpointing

Both phases append to the volume as results land. Re-running skips whatever
is already recorded. Checkpoint files are keyed to the question-set run id,
so they persist across notebook restarts.

In [ ]:
CKPT_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, "_checkpoint")
os.makedirs(CKPT_DIR, exist_ok=True)


class Checkpoint:
    """Append-only JSONL on the volume. Thread-safe, flushes every write."""

    def __init__(self, path, key_fn):
        self.path, self.key_fn = path, key_fn
        self.lock = threading.Lock()
        self.rows, self.done = [], set()
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        continue          # tolerate a torn final line
                    self.rows.append(r)
                    self.done.add(key_fn(r))
        self.fh = open(path, "a")

    def has(self, key):
        return key in self.done

    def add(self, row):
        with self.lock:
            self.fh.write(json.dumps(row) + "\n")
            self.fh.flush()
            os.fsync(self.fh.fileno())
            self.rows.append(row)
            self.done.add(self.key_fn(row))

    def close(self):
        try:
            self.fh.close()
        except Exception:
            pass


eval_ckpt = Checkpoint(os.path.join(CKPT_DIR, f"answers_{rid}.jsonl"),
                       lambda r: (r["config_id"], r["question_id"]))
judge_ckpt = Checkpoint(os.path.join(CKPT_DIR, f"grades_{rid}.jsonl"),
                        lambda r: (r["judge"], r["config_id"], r["question_id"]))

print(f"Eval checkpoint : {len(eval_ckpt.rows):,} rows already recorded")
print(f"Judge checkpoint: {len(judge_ckpt.rows):,} rows already recorded")
print(f"\nDelete these files to start clean:")
print(f"  {eval_ckpt.path}")
print(f"  {judge_ckpt.path}")

## Request helper

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def guarded_post(body, headers, timeout=REQUEST_TIMEOUT, purpose="call"):
    """One POST to OpenRouter. The attempt is reserved at its worst case before it is sent and settled after, so
    retries, exceptions and rejected calls are all counted. BudgetExceeded is raised before anything is sent."""
    body = openrouter_body(body)
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = requests.post(OPENROUTER_URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 200:
            try:
                usage = (r.json() or {}).get("usage") or None
            except Exception:
                usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


def call_openrouter(body, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES, retry_on_empty=False):
    last = None
    for attempt in range(retries):
        try:
            r = guarded_post(body, HEADERS, timeout=timeout, purpose="call")
            if r.status_code in (429, 503):
                w = r.headers.get("Retry-After")
                time.sleep(float(w) if w and w.isdigit() else 2 ** (attempt + 2))
                last = f"HTTP {r.status_code}"
                continue
            if r.status_code >= 400:
                last = f"HTTP {r.status_code}: {r.text[:300]}"
                if 400 <= r.status_code < 500 and r.status_code != 429:
                    return {"ok": False, "error": last}
                time.sleep(2 ** attempt)
                continue
            d = r.json()
            if "error" in d:
                return {"ok": False, "error": str(d["error"])[:300]}
            text = (d["choices"][0]["message"].get("content") or "").strip()
            if not text and retry_on_empty and attempt < retries - 1:
                last = "empty content"
                time.sleep(1 + attempt)
                continue
            u = d.get("usage", {}) or {}
            return {"ok": True, "text": text, "provider": d.get("provider"),
                    "is_byok": u.get("is_byok"),
                    "prompt_tokens": u.get("prompt_tokens", 0),
                    "completion_tokens": u.get("completion_tokens", 0),
                    "reasoning_tokens": (u.get("completion_tokens_details") or {}).get("reasoning_tokens", 0),
                    "error": None}
        except BudgetExceeded:
            raise
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:300]}"
            time.sleep(2 ** attempt)
    return {"ok": False, "error": last}


def token_cost(model, pin, pout):
    p = PRICE.get(model)
    return 0.0 if not p else pin / 1e6 * p[0] + pout / 1e6 * p[1]


print("helper loaded")

## Probes

In [ ]:
for c in CONFIGS:
    body = {"model": c["model"], "provider": EVAL_PROVIDER,
            "messages": [{"role": "user", "content": "Reply with the single word OK."}],
            "max_tokens": 2000, "reasoning": {"effort": c["effort"]}}
    if c["mode"]:
        body["reasoning"]["mode"] = c["mode"]
    res = call_openrouter(body, timeout=180, retries=2)
    print(f"[{'OK  ' if res['ok'] else 'FAIL'}] {c['config_id']:<24} "
          f"provider={res.get('provider')} byok={res.get('is_byok')} "
          f"{'' if res['ok'] else str(res.get('error'))[:110]}")

print()
for name in USE_JUDGES:
    j = JUDGES[name]
    res = call_openrouter({"model": j["slug"], "provider": j["provider"],
                           "messages": [{"role": "user", "content": "Reply with the single word OK."}],
                           "max_tokens": 2000}, timeout=180, retries=2)
    print(f"[{'OK  ' if res['ok'] else 'FAIL'}] {name:<8} provider={res.get('provider')} "
          f"byok={res.get('is_byok')} {'' if res['ok'] else str(res.get('error'))[:110]}")

## Answer generation (checkpointed)

In [ ]:
ANSWER_SYSTEM = """You are a food safety regulatory analyst. Answer the question directly and precisely.

Cite the specific CFR subsection your answer rests on. State any condition or exception that changes the answer. If the regulation does not address the question, say so rather than inferring a requirement.

Keep the answer under 200 words. No preamble."""


def run_eval_task(task):
    cfg, q = task
    body = {"model": cfg["model"], "provider": EVAL_PROVIDER,
            "messages": [{"role": "system", "content": ANSWER_SYSTEM},
                         {"role": "user", "content": q["question"]}],
            "max_tokens": MAX_TOKENS, "reasoning": {"effort": cfg["effort"]}}
    if cfg["mode"]:
        body["reasoning"]["mode"] = cfg["mode"]
    res = call_openrouter(body)
    row = {"config_id": cfg["config_id"], "model": cfg["model"], "effort": cfg["effort"],
           "question_id": q["id"], "question": q["question"],
           "reference_answer": q["reference_answer"], "reference_citation": q.get("citation", ""),
           "domain": q["domain"], "difficulty": q.get("difficulty"),
           "seed_section": q["seed_section"],
           "answer": res.get("text", ""), "ok": res["ok"], "error": res.get("error"),
           "is_byok": res.get("is_byok"), "provider": res.get("provider"),
           "prompt_tokens": res.get("prompt_tokens", 0),
           "completion_tokens": res.get("completion_tokens", 0),
           "reasoning_tokens": res.get("reasoning_tokens", 0),
           "cost_computed": token_cost(cfg["model"], res.get("prompt_tokens", 0),
                                       res.get("completion_tokens", 0))}
    eval_ckpt.add(row)
    return row


TASKS = [(c, q) for c in CONFIGS for q in EVAL_QUESTIONS]
remaining = [(c, q) for c, q in TASKS if not eval_ckpt.has((c["config_id"], q["id"]))]

run_id = RUN_ID
print(f"Run {run_id}")
print(f"  total tasks : {len(TASKS):,}")
print(f"  already done: {len(TASKS) - len(remaining):,}")
print(f"  remaining   : {len(remaining):,}", flush=True)

_t0 = time.time(); _n = [0]


def _tracked(t):
    r = run_eval_task(t)
    _n[0] += 1
    if _n[0] % 50 == 0:
        el = time.time() - _t0
        print(f"  {_n[0]:,}/{len(remaining):,}  eta {(len(remaining)-_n[0])/(_n[0]/el)/60:.0f}m", flush=True)
    return r


if remaining:
    with concurrent.futures.ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        list(pool.map(_tracked, remaining))

eval_elapsed = time.time() - _t0
ANSWERS = eval_ckpt.rows
ok_answers = [a for a in ANSWERS if a["ok"] and a["answer"].strip()]
print(f"\nDone in {eval_elapsed/60:.1f} min")
print(f"  answers on disk : {len(ANSWERS):,}")
print(f"  usable          : {len(ok_answers):,}")
print(f"  cost this run   : ${sum(a['cost_computed'] for a in ANSWERS):.2f}")
print(f"  is_byok         : {dict(Counter(a['is_byok'] for a in ANSWERS))}")

## Judging against full CFR text (checkpointed)

The regulation itself is the ground truth here. The reference answer is
provided as a second opinion and the judge is told it may be incomplete.

In [ ]:
JUDGE_SYSTEM = """You grade answers to US food safety regulation questions. You are given the FULL TEXT of the governing CFR section. That text is the ground truth. A short reference answer is also provided, but it is a summary and may omit relevant provisions — if the answer and the reference disagree, judge against the CFR text, not the reference.

accuracy (0-5): Is the answer correct as a statement of what this section requires?
  5 = correct and precise; a regulatory professional would sign it
  4 = correct, minor imprecision that does not change the obligation
  3 = substantially correct, one meaningful gap or vague point
  2 = partially correct, a material error alongside correct content
  1 = mostly wrong
  0 = wrong, or answers a different question

citation (0-3): Does it point to the right place in the regulation?
  3 = correct subsection, precisely identified
  2 = correct section, subsection missing or slightly off
  1 = right part, wrong section
  0 = no citation, or unrelated

completeness (0-3): Does it capture the conditions, exceptions and qualifiers in the section that bear on the question?

contradiction (boolean): Does the answer state something that CONTRADICTS the CFR text, or assert a requirement, deadline or threshold the section does not contain?
  Citing a related provision outside this section is NOT a contradiction.
  Adding correct context is NOT a contradiction.
  Only flag a direct conflict with the text or an invented requirement.

Return JSON only:
{"accuracy": N, "citation": N, "completeness": N, "contradiction": true|false, "note": "one short sentence"}"""

JUDGE_TEMPLATE = """=== GOVERNING CFR SECTION: 21 CFR {section} (ground truth) ===
{cfr_text}
=== END SECTION ===

QUESTION:
{question}

REFERENCE ANSWER (a summary; may be incomplete):
{reference}

ANSWER TO GRADE:
{answer}

Grade against the CFR text above. JSON only."""


def judge_one(args):
    jn, ans = args
    j = JUDGES[jn]
    cfr = CFR_TEXT.get(ans["seed_section"], "")[:CFR_TEXT_CAP]
    body = {"model": j["slug"], "provider": j["provider"],
            "messages": [{"role": "system", "content": JUDGE_SYSTEM},
                         {"role": "user", "content": JUDGE_TEMPLATE.format(
                             section=ans["seed_section"], cfr_text=cfr,
                             question=ans["question"], reference=ans["reference_answer"],
                             answer=ans["answer"][:4000])}],
            "max_tokens": 4000}
    res = call_openrouter(body, timeout=240, retry_on_empty=True)
    row = {"judge": jn, "config_id": ans["config_id"], "question_id": ans["question_id"],
           "seed_section": ans["seed_section"], "domain": ans["domain"],
           "difficulty": ans["difficulty"],
           "judge_prompt_tokens": res.get("prompt_tokens", 0),
           "judge_completion_tokens": res.get("completion_tokens", 0)}
    if not res["ok"] or not res["text"]:
        row.update({"accuracy": None, "error": res.get("error") or "empty"})
    else:
        try:
            txt = res["text"].replace("```json", "").replace("```", "").strip()
            b = txt.find("{")
            if b > 0:
                txt = txt[b:]
            g = json.loads(txt)
            acc, cite = int(g.get("accuracy", 0)), int(g.get("citation", 0))
            row.update({"accuracy": acc, "citation": cite,
                        "completeness": int(g.get("completeness", 0)),
                        "contradiction": bool(g.get("contradiction")),
                        "note": g.get("note", ""),
                        "weighted": round((acc / ACC_MAX) * W_ACCURACY * 10
                                          + (cite / CITE_MAX) * W_CITATION * 10, 3),
                        "error": None})
        except Exception as e:
            row.update({"accuracy": None,
                        "error": f"parse: {type(e).__name__} :: {res['text'][:120]}"})
    judge_ckpt.add(row)
    return row


judge_tasks = [(jn, a) for jn in USE_JUDGES for a in ok_answers
               if not judge_ckpt.has((jn, a["config_id"], a["question_id"]))]

print(f"Judge tasks total  : {len(USE_JUDGES) * len(ok_answers):,}")
print(f"Already checkpointed: {len(judge_ckpt.rows):,}")
print(f"Remaining          : {len(judge_tasks):,}", flush=True)

_j0 = time.time(); _jn = [0]


def _tj(t):
    r = judge_one(t)
    _jn[0] += 1
    if _jn[0] % 100 == 0:
        el = time.time() - _j0
        print(f"  {_jn[0]:,}/{len(judge_tasks):,}  eta {(len(judge_tasks)-_jn[0])/(_jn[0]/el)/60:.0f}m", flush=True)
    return r


if judge_tasks:
    with concurrent.futures.ThreadPoolExecutor(max_workers=JUDGE_WORKERS) as pool:
        list(pool.map(_tj, judge_tasks))

judge_elapsed = time.time() - _j0
GRADES = judge_ckpt.rows
graded = [g for g in GRADES if g.get("accuracy") is not None]
print(f"\nDone in {judge_elapsed/60:.1f} min")
print(f"  gradings on disk: {len(GRADES):,}")
print(f"  parsed          : {len(graded):,}")
_bad = [g for g in GRADES if g.get("accuracy") is None]
if _bad:
    print(f"  failed          : {dict(Counter(g['judge'] for g in _bad))}")

## Scoring

In [ ]:
def bootstrap_ci(vals, n=2000, alpha=0.05):
    if len(vals) < 2:
        return (None, None)
    rng = random.Random(SEED)
    m = [sum(vals[rng.randrange(len(vals))] for _ in range(len(vals))) / len(vals)
         for _ in range(n)]
    m.sort()
    return round(m[int(n * alpha / 2)], 3), round(m[int(n * (1 - alpha / 2))], 3)


paired = defaultdict(dict)
for g in graded:
    paired[(g["config_id"], g["question_id"])][g["judge"]] = g["accuracy"]
both = [(v[USE_JUDGES[0]], v[USE_JUDGES[1]]) for v in paired.values() if len(v) == 2]

print("=" * 76)
if both:
    d = [abs(a - b) for a, b in both]
    try:
        corr = statistics.correlation([x[0] for x in both], [x[1] for x in both])
    except Exception:
        corr = None
    print("JUDGE AGREEMENT ON ACCURACY (0-5)")
    print(f"  paired         : {len(both):,}")
    print(f"  exact          : {sum(1 for x in d if x == 0)/len(d):.1%}")
    print(f"  within 1       : {sum(1 for x in d if x <= 1)/len(d):.1%}")
    print(f"  mean |diff|    : {statistics.mean(d):.2f} of 5")
    print(f"  correlation    : {corr if corr is None else round(corr,3)}")
    print(f"  {USE_JUDGES[0]} mean {statistics.mean(x[0] for x in both):.2f}  |  "
          f"{USE_JUDGES[1]} mean {statistics.mean(x[1] for x in both):.2f}")
else:
    corr = None

agg = defaultdict(lambda: defaultdict(list))
for g in graded:
    c = g["config_id"]
    agg[c]["acc"].append(g["accuracy"])
    agg[c]["cite"].append(g["citation"])
    agg[c]["comp"].append(g["completeness"])
    agg[c]["w"].append(g["weighted"])
    agg[c]["contra"].append(1 if g.get("contradiction") else 0)

cost_by = defaultdict(float); cnt_by = defaultdict(int)
for a in ANSWERS:
    cost_by[a["config_id"]] += a["cost_computed"]
    cnt_by[a["config_id"]] += 1

rows = []
for c, v in agg.items():
    lo, hi = bootstrap_ci(v["acc"])
    wlo, whi = bootstrap_ci(v["w"])
    cpq = cost_by[c] / max(cnt_by[c], 1)
    rows.append({"config_id": c, "n": len(v["acc"]),
                 "accuracy": round(statistics.mean(v["acc"]), 3),
                 "acc_ci_low": lo, "acc_ci_high": hi,
                 "citation": round(statistics.mean(v["cite"]), 3),
                 "completeness": round(statistics.mean(v["comp"]), 3),
                 "weighted": round(statistics.mean(v["w"]), 3),
                 "w_ci_low": wlo, "w_ci_high": whi,
                 "contradiction_rate": round(statistics.mean(v["contra"]), 4),
                 "cost_per_q": round(cpq, 5), "cost_15k": round(cpq * 15000, 0)})
rows.sort(key=lambda r: -r["accuracy"])

print("\n" + "=" * 76)
print("RANKED BY ACCURACY (0-5), judged against full CFR text")
print("=" * 76)
print(f"{'config':<24}{'acc':>6}{'95% CI':>16}{'cite':>6}{'comp':>6}{'wtd':>7}{'contra':>8}{'$15k':>8}")
for r in rows:
    ci = f"[{r['acc_ci_low']}, {r['acc_ci_high']}]"
    print(f"{r['config_id']:<24}{r['accuracy']:>6.3f}{ci:>16}{r['citation']:>6.2f}"
          f"{r['completeness']:>6.2f}{r['weighted']:>7.2f}"
          f"{r['contradiction_rate']*100:>7.1f}%{r['cost_15k']:>8,.0f}")

top = rows[0]
ov = [r for r in rows[1:] if r["acc_ci_high"] is not None and r["acc_ci_high"] >= top["acc_ci_low"]]
print(f"\nAccuracy leader: {top['config_id']} ({top['accuracy']:.3f})")
print(f"CI overlaps leader: {len(ov)} of {len(rows)-1}")

print("\nContradiction rate — the filter that matters for a teacher:")
for r in sorted(rows, key=lambda x: x["contradiction_rate"]):
    bad = r["contradiction_rate"] * 15000
    print(f"  {r['config_id']:<24} {r['contradiction_rate']*100:>5.1f}%   "
          f"~{bad:,.0f} flawed pairs per 15,000")

print("\nAccuracy by difficulty tier:")
bd = defaultdict(lambda: defaultdict(list))
for g in graded:
    bd[g["config_id"]][g["difficulty"]].append(g["accuracy"])
tiers = ["recall", "application", "reconciliation"]
print(f"{'config':<24}" + "".join(f"{t:>16}" for t in tiers))
for r in rows:
    line = f"{r['config_id']:<24}"
    for t in tiers:
        v = bd[r["config_id"]].get(t, [])
        line += f"{statistics.mean(v):>16.3f}" if v else f"{'—':>16}"
    print(line)

## Write final outputs

In [ ]:
outdir = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, run_id)
os.makedirs(outdir, exist_ok=True)


def dump(name, data):
    p = os.path.join(outdir, name)
    with open(p, "w") as f:
        for r in data:
            f.write(json.dumps(r) + "\n")
    return p


manifest = {
    "run_id": run_id, "stage": "2c_confirm",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "question_run_id": rid, "screen_run_id": srid,
    "questions_used": len(EVAL_QUESTIONS),
    "question_ids": sorted(q["id"] for q in EVAL_QUESTIONS),
    "set_b_size": len(B_IDS), "set_b_touched": False,
    "configs": [c["config_id"] for c in CONFIGS],
    "judged_against": "full CFR section text",
    "cfr_text_cap": CFR_TEXT_CAP,
    "weights": {"accuracy": W_ACCURACY, "citation": W_CITATION,
                "accuracy_max": ACC_MAX, "citation_max": CITE_MAX},
    "eval_calls": len(ANSWERS), "eval_ok": len(ok_answers),
    "judges": USE_JUDGES, "judge_calls": len(GRADES), "judge_ok": len(graded),
    "judge_agreement_accuracy": {
        "paired": len(both),
        "correlation": round(corr, 4) if both and corr is not None else None,
    },
    "eval_cost_usd": round(sum(a["cost_computed"] for a in ANSWERS), 2),
    "elapsed_min": {"eval": round(eval_elapsed / 60, 1), "judge": round(judge_elapsed / 60, 1)},
    "ranking": rows,
    "checkpoints": {"answers": eval_ckpt.path, "grades": judge_ckpt.path},
    "files": {"answers": dump("answers.jsonl", ANSWERS),
              "grades": dump("grades.jsonl", GRADES),
              "results": dump("results.jsonl", rows)},
}
with open(os.path.join(outdir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

eval_ckpt.close()
judge_ckpt.close()

print(f"Eval cost : ${manifest['eval_cost_usd']:.2f}")
print(f"Wall time : {manifest['elapsed_min']['eval']:.0f}m eval + "
      f"{manifest['elapsed_min']['judge']:.0f}m judge")
print(f"Output    : {outdir}")
print(f"\nSet B ({len(B_IDS)}) still sealed for the final evaluation.")

## Inspect

Read the low-accuracy gradings before choosing. A judge misreading the CFR
text shows up here and nowhere else.

In [ ]:
def show(config_id=None, n=3, lowest=True, contradictions_only=False):
    akey = {(a["config_id"], a["question_id"]): a for a in ok_answers}
    pool_ = [g for g in graded if config_id is None or g["config_id"] == config_id]
    if contradictions_only:
        pool_ = [g for g in pool_ if g.get("contradiction")]
    if lowest:
        pool_ = sorted(pool_, key=lambda g: g["accuracy"])
    for g in pool_[:n]:
        a = akey.get((g["config_id"], g["question_id"]))
        if not a:
            continue
        print("=" * 76)
        print(f"{g['config_id']} | {g['question_id']} | {g['judge']} | "
              f"acc={g['accuracy']}/5 cite={g['citation']}/3 contra={g.get('contradiction')}")
        print(f"  note: {g.get('note')}")
        print(f"\nQ: {a['question'][:280]}")
        print(f"\nREF: {a['reference_answer'][:300]}")
        print(f"\nGOT: {a['answer'][:400]}\n")


if graded:
    print("Flagged contradictions — verify these are real before trusting the rate:\n")
    show(n=3, lowest=False, contradictions_only=True)